In [ ]:
### INSTALLATION CELL ###
!pip install -qU numpy==1.26.4 PyMuPDF langchain langchain-community langchain-groq gradio sentence-transformers faiss-gpu torch torchvision torchaudio

import os
import uuid
import warnings
import gradio as gr
import fitz 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.storage import InMemoryStore
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.documents import Document
import faiss

warnings.filterwarnings("ignore")

os.environ["GROQ_API_KEY"] = "API_KEY"


In [2]:

# RAG Pipeline Functions

def load_docs_by_page(file_path):
    """Extracts text and creates a Document object for each page."""
    docs = []
    with fitz.open(file_path) as doc:
        for i, page in enumerate(doc):
            docs.append(Document(
                page_content=page.get_text(),
                metadata={'page': i + 1}
            ))
    return docs

def create_summary_chain():
    """Creates a chain to generate summaries for document chunks."""
    summary_prompt_template = """
    Summarize the following text from a project report in 2-3 sentences.
    Capture the main topic and key points.
    Text: "{text}"
    Summary:
    """
    prompt = ChatPromptTemplate.from_template(summary_prompt_template)
    model = ChatGroq(model_name="llama-3.3-70b-versatile")
    return {"text": RunnablePassthrough()} | prompt | model | StrOutputParser()

def setup_multi_vector_retriever(docs):
    """Sets up the MultiVectorRetriever with summaries."""
    docstore = InMemoryStore()
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={'device': 'cuda'}
    )
    embedding_dim = 384
    index = faiss.IndexFlatL2(embedding_dim)
    vectorstore = FAISS(
        embedding_function=embeddings,
        index=index,
        docstore=InMemoryDocstore(),
        index_to_docstore_id={}
    )

    # Create the retriever
    retriever = MultiVectorRetriever(
        vectorstore=vectorstore,
        docstore=docstore,
        id_key="doc_id",
    )

    doc_ids = [str(uuid.uuid4()) for _ in docs]

    chain = create_summary_chain()
    
    summaries = chain.batch(docs, {"max_concurrency": 5})

    summary_docs = [
        Document(page_content=s, metadata={"doc_id": doc_ids[i]})
        for i, s in enumerate(summaries)
    ]
    
    retriever.vectorstore.add_documents(summary_docs)
    retriever.docstore.mset(list(zip(doc_ids, docs)))
    
    return retriever



In [3]:
def format_docs(docs):
    """Helper function to format retrieved documents for the prompt."""
    return "\n\n".join([doc.page_content for doc in docs])

def create_rag_chain(retriever):
    """Creates the complete RAG chain for question-answering."""
    prompt_template_str = """
        You are an assistant for question-answering tasks. Use the following pieces of retrieved context from a project report to answer the question accurately and thoroughly.
        If the context does not contain the answer, state that the information is not available in the provided document.
        Answer in clear, well-structured bullet points.

        Question: {question}
        Context: {context}
        Answer:
    """
    model = ChatGroq(model_name="llama-3.3-70b-versatile") # Using a more powerful model for final generation
    prompt = ChatPromptTemplate.from_template(prompt_template_str)
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | model
        | StrOutputParser()
    )
    return rag_chain



In [ ]:
# --- Gradio Application Logic ---

rag_chain = None

def upload_and_process(file):
    """Handles file upload, processing, and RAG chain creation."""
    global rag_chain
    if file is None:
        return "❌ Please upload a file first."
    try:
        file_path = file.name
        page_docs = load_docs_by_page(file_path)
        
        # This step will now be slower as it generates summaries
        retriever = setup_multi_vector_retriever(page_docs)
        
        rag_chain = create_rag_chain(retriever)
        return f"✅ Document processed into {len(page_docs)} pages. Multi-vector retrieval is ready."
    except Exception as e:
        return f"❌ Error processing file: {e}"

def ask_question(question):
    """Handles user questions and streams the response."""
    if rag_chain is None:
        return "❌ Please upload and process a document first."
    if not question:
        return "🤔 Please enter a question."
    try:
        response = ""
        for chunk in rag_chain.stream(question):
            response += chunk
        return response
    except Exception as e:
        print(f"An error occurred during the RAG chain stream: {e}")
        return f"❌ An error occurred. Please check the console for details. Error: {e}"

# --- Gradio Interface Definition ---

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("## 🧠 Advanced RAG QA with Summaries")
    with gr.Row():
        with gr.Column():
            file_input = gr.File(label="📁 Upload PDF Document", file_types=[".pdf"])
            upload_btn = gr.Button("🔄 Upload & Process")
            upload_output = gr.Textbox(label="Processing Status", interactive=False)
        with gr.Column():
            question_input = gr.Textbox(label="❓ Ask a Question", placeholder="e.g., What are the conclusions and results of this project?")
            ask_btn = gr.Button("🔍 Get Answer")
            answer_output = gr.Textbox(label="📢 Answer", lines=10, interactive=False)

    upload_btn.click(fn=upload_and_process, inputs=[file_input], outputs=[upload_output])
    ask_btn.click(fn=ask_question, inputs=[question_input], outputs=[answer_output])

# --- Launch the Application ---
print("Attempting to launch Gradio interface... 🚀")
demo.launch(server_name="0.0.0.0", share=True)

Attempting to launch Gradio interface... 🚀
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://d2d3a8b275c0a5cc2c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


2025-10-01 11:48:36.881961: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
